# What Should Your AI Agent Actually Remember? 3 Ways to Build Selective Memory

A real conversation mixes durable facts ("I'm vegetarian"), throwaway small talk ("gorgeous weather!"), preferences ("I refuse overnight layovers"), and events ("I just booked the Madrid flight"). **Store everything** and memory gets expensive and dirty — [Demo 05](../05-memory-hygiene-demo/) shows it's also dangerous. **Store nothing** and you're back to the amnesiac agent of Demo 01.

The missing capability is **selection**: deciding what deserves to persist, in which memory *type*, and what to ignore. This notebook builds it three ways and measures them against the same planted conversation:

| Mechanism | Who selects | Where it stores | Runs |
|-----------|-------------|-----------------|------|
| **A — agent tools** | the conversational agent itself | `agent.state` (key-value) | inline, inside the turn |
| **B — own extractor** | 4 specialized LLM prompts (one per type) | Amazon S3 Vectors, one index per type | off the conversation path |
| **C — AgentCore Memory** | 4 built-in managed strategies | AgentCore's managed store | async, inside AWS |

B is "AgentCore built by hand" — the same pipeline (extract per type → embed → index per type), except you own the prompts and pay the tokens. C is the managed version: send raw turns, AWS does the rest.

The ground truth is planted: **5 items that must be kept** (2 facts, 2 preferences, 1 episode) and **3 decoys that must not** (small talk, a passing opinion, ephemeral weather). Scoring is deterministic — no LLM judge.

This demo uses Strands Agents. The pattern carries over to other agent frameworks.

## Install dependencies

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Configure credentials

- **AWS credentials** (`aws configure`) — Titan embeddings, S3 Vectors, and AgentCore Memory. All resources are **created automatically** if missing (4 vector indexes + 1 AgentCore memory).
- **`OPENAI_API_KEY`** — the conversational agent (A) and the extractor (B).

In [2]:
import os

# Bearer-token env vars would override the AWS profile — drop them before boto3 loads.
os.environ.pop('AWS_BEARER_TOKEN', None)
os.environ.pop('AWS_BEARER_TOKEN_BEDROCK', None)

from dotenv import load_dotenv
load_dotenv()

assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env'

## The planted conversation and its ground truth

Six turns from a brand-new user. Five plant things worth keeping; three are decoys that a good selector must ignore. Every mechanism receives exactly these turns.

In [3]:
import json, time

CONVERSATION = [
    "Hi! I'm Sam. I'm vegetarian with a severe shellfish allergy.",              # keep: 2 facts
    "Gorgeous weather out here today, hope your day is going great!",            # decoy: small talk
    "For flights: I refuse overnight layovers, and I keep fares under $1,500.",  # keep: 2 preferences
    "I watched a documentary about airplanes last night, it was okay I guess.",  # decoy: passing opinion
    "I just booked the Iberia flight JFK to Madrid for October 10th!",           # keep: 1 episode
    "Apparently it might drizzle here later this afternoon.",                    # decoy: ephemeral
]

KEEPERS = {'vegetarian': 'F1', 'shellfish': 'F2', 'layover': 'P1', '1,500': 'P2', 'madrid': 'E1'}
DECOYS = {'gorgeous': 'D1', 'documentary': 'D2', 'drizzle': 'D3'}

def score(text):
    text = text.lower()
    kept = [tag for marker, tag in KEEPERS.items() if marker in text]
    leaked = [tag for marker, tag in DECOYS.items() if marker in text]
    return f'kept {len(kept)}/5 {sorted(kept)} | decoys leaked: {leaked or "none"}'

---
## Mechanism A — the agent selects, inline

The typed-memory prompt + four generic tools (from the earlier version of this demo). Selection happens **inside the conversational turn**: the same model that chats decides what to file. Zero extra infrastructure — and the selection cost rides on every turn's latency.

In [4]:
# OTEL_SDK_DISABLED silences OpenTelemetry tracing noise in notebook output.
os.environ['OTEL_SDK_DISABLED'] = 'true'

# Agent is the Strands agent loop: model + tools until the answer is done.
from strands import Agent

# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel

# The four generic memory tools the agent files sections with.
from tools import core_memory_read, core_memory_write, core_memory_update, core_memory_list

MODEL = OpenAIModel(model_id='gpt-4o-mini')

# Amazon Bedrock instead: comment above, uncomment below.
# from strands.models import BedrockModel
# MODEL = BedrockModel(model_id='openai.gpt-oss-120b-1:0', region_name='us-west-2')

SYSTEM_PROMPT = (
    "You are a flight assistant with self-managed memory. "
    "Organize what you learn into these memory sections:\n"
    "- 'facts': durable facts about the user's world (name, home airport, allergies)\n"
    "- 'preferences': likes/dislikes they reveal (cabin, layovers, budget)\n"
    "- 'trip_summary': one rolling summary of the trip being planned\n"
    "- 'episodes': notable events, one entry per event (bookings, cancellations)\n"
    "Only store what is durable — ignore small talk and passing remarks. "
    "Be concise — answer in 2-3 sentences maximum."
)

agent = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT,
              tools=[core_memory_read, core_memory_write, core_memory_update, core_memory_list],
              callback_handler=None)

turn_ms = []
for turn in CONVERSATION:
    t0 = time.perf_counter()
    agent(turn)
    turn_ms.append((time.perf_counter() - t0) * 1000)

memory_a = agent.state.get('core_memory') or {}
print('sections:', sorted(memory_a))
print(score(json.dumps(memory_a)))
print(f'avg turn latency (selection inline): {sum(turn_ms)/len(turn_ms):.0f} ms')

sections: ['episodes', 'facts', 'preferences']
kept 4/5 ['E1', 'F1', 'F2', 'P1'] | decoys leaked: none
avg turn latency (selection inline): 1667 ms


---
## Mechanism B — your own extractor, off the conversation path

Four **specialized extraction prompts** — one per memory type — run over each raw turn *after* the conversation (in production: a hook or async worker). Each decides independently: extract, or `NOTHING`. Survivors are embedded (Titan V2) and written to **S3 Vectors, one index per type** — AgentCore's per-strategy partitioning, built by hand.

The conversational agent never sees any of this. What you own: the selection criteria (the prompts). What you pay: extractor tokens per turn.

In [5]:
# extractor holds the 4 prompts + the typed S3 Vectors store (self-provisioned).
import extractor

typed = extractor.TypedVectorMemory()   # creates the 4 indexes if missing
typed.clear()                           # rerun-safe

total_tokens, pipeline_ms = 0, []
for n, turn in enumerate(CONVERSATION):
    kept, cost = extractor.extract(turn)
    total_tokens += cost['tokens']
    write_ms = sum(typed.write(t, items, f'turn{n}') for t, items in kept.items())
    pipeline_ms.append(cost['ms'] + write_ms)
    if kept:
        print(f'turn {n}: kept {list(kept)}')
    else:
        print(f'turn {n}: NOTHING (decoy filtered)')

print()
print('stored per type:', typed.counts())
stored = ' '.join(t for mt in extractor.INDEXES for t, _ in typed.query(mt, 'traveler profile and trip', 10))
print(score(stored))
print(f'extractor cost: {total_tokens:,} tokens | avg availability lag: {sum(pipeline_ms)/len(pipeline_ms)/1000:.1f}s per turn')

turn 0: kept ['facts', 'summary']


turn 1: NOTHING (decoy filtered)


turn 2: kept ['preferences', 'summary']


turn 3: kept ['summary']


turn 4: kept ['facts', 'summary', 'episodes']


turn 5: kept ['summary']



stored per type: {'facts': 7, 'preferences': 2, 'summary': 5, 'episodes': 1}


kept 5/5 ['E1', 'F1', 'F2', 'P1', 'P2'] | decoys leaked: none
extractor cost: 1,999 tokens | avg availability lag: 4.6s per turn


---
## Mechanism C — AgentCore Memory: the managed pipeline

Send the **raw turns** (`create_event`) — no selection on your side at all. The four built-in strategies (semantic / userPreference / summary / episodic) extract, embed, and index asynchronously inside AWS. The memory (with all 4 strategies) is created by `ensure_memory()` if it doesn't exist.

Two API facts learned by running this (not in the docs): the episodic strategy **requires** `reflectionConfiguration.namespaces` — a bare `{'name': ...}` fails validation — and a memory in `CREATING` status can't be deleted; wait for `ACTIVE`.

Extraction is async, so we poll and **measure the lag** — a number AWS doesn't publish.

In [6]:
# agentcore_memory wraps the two AgentCore clients + self-provisioning.
import agentcore_memory as acm

info = acm.ensure_memory()      # creates the memory with 4 strategies if missing
memory_id, sids = info['memory_id'], info['strategy_ids']
actor = f'sam-{int(time.time())}'   # fresh actor -> clean namespaces per run
session = 'selective-run'

turn_ms_c = [acm.send_turn(memory_id, actor, session, 'USER', t) for t in CONVERSATION]
print(f'6 raw turns sent | create_event avg: {sum(turn_ms_c)/len(turn_ms_c):.0f} ms')

lag = acm.wait_for_extraction(memory_id, sids['facts'], actor,
                              'dietary restrictions travel preferences', timeout_s=420)
print(f'extraction lag (measured): {lag:.0f}s')

stored_c, per_strategy = '', {}
for name, sid in sids.items():
    scoped = name in ('tripSummary', 'episodes')
    records = acm.retrieve(memory_id, sid, actor, 'traveler profile and trip',
                           session_id=session if scoped else None, top_k=10)
    per_strategy[name] = len(records)
    stored_c += ' '.join(records) + ' '

print('records per strategy:', per_strategy)
print(score(stored_c))

6 raw turns sent | create_event avg: 347 ms


extraction lag (measured): 85s


records per strategy: {'episodes': 0, 'facts': 1, 'preferences': 2, 'tripSummary': 1}
kept 5/5 ['E1', 'F1', 'F2', 'P1', 'P2'] | decoys leaked: ['D2', 'D3']


---
## The measured comparison

Numbers from a real run of this notebook (yours will vary some — extraction is nondeterministic):

| Mechanism | Kept | Decoys leaked | Turn overhead | Available after | Extra cost |
|-----------|------|---------------|---------------|-----------------|------------|
| A — agent tools (inline) | 4/5 | 0 | ~1.7 s/turn (selection inside the turn) | immediately | none |
| B — own extractor + S3V | **5/5** | 0 | **0 — off-path** | ~4 s/turn | ~2k tokens / 6 turns |
| C — AgentCore managed | 4/5 | 1 | ~0.4 s/turn (`create_event` only) | **~53 s** (measured) | managed pricing |

What the numbers teach:

- **A** is free but couples selection to conversation latency, and its filing quality rides on the chat model's attention.
- **B** won on selection quality here — specialized single-purpose prompts beat both a multitasking agent and the generic managed extractor — and it keeps the conversation path untouched. The price: you own 4 prompts and pay their tokens.
- **C** has the cheapest write path (raw `create_event`, no LLM on your side) and zero pipeline to maintain, but extraction lands ~a minute later and the built-in criteria aren't yours to tune (it kept a decoy).

## The decision table

| Situation | Mechanism |
|-----------|-----------|
| Prototype; want the agent's memory decisions visible in the conversation | **A** — tools inline |
| You need control of the keep/discard criteria (regulated domain, custom taxonomy) — or your own storage | **B** — own extractor |
| Production multi-user; a minute of extraction lag is fine; no pipeline to maintain | **C** — AgentCore strategies |

**Orthogonal to backends:** selection decides *what* enters; the backend decides *where* it lives. A writes to key-value today, B to S3 Vectors — either output could target the graph of [Demo 03](../03-graph-memory-demo/) (facts as triples). C bundles pipeline + storage together — that's part of its trade-off.

**Next:** [Demo 05 — Memory Hygiene](../05-memory-hygiene-demo/) covers the flip side of selection: what an agent must NOT remember, and how to evict poison that got in.